# LoRA v2 — TTA (Test-Time Augmentation) with Choice Permutation

Shuffles choice order K times, runs inference each time, maps scores back to original positions, averages. Eliminates position bias. No training — uses v2 checkpoint.

In [ ]:
!pip install -q "transformers==4.57.1" "peft==0.18.0" bitsandbytes accelerate "pandas==2.2.2" "pillow==11.3.0" tqdm

In [ ]:
import kagglehub
from pathlib import Path
kagglehub.login()
COMP_DIR = Path(kagglehub.competition_download("pixels-to-predictions"))
print("Downloaded to:", COMP_DIR)

In [ ]:
import os, json, random, gc
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq
from peft import PeftModel

SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
CHOICE_LETTERS = "ABCDEFGHIJ"
BATCH_SIZE_INFER = 4

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
DATA_DIR = COMP_DIR / "data"
if not DATA_DIR.exists(): DATA_DIR = COMP_DIR

train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

example_name = Path(val_df.iloc[0]["image_path"]).name
matches = list(COMP_DIR.rglob(example_name))
DATA_ROOT = matches[0].parents[2]
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

In [ ]:
def clean_text(x):
    if pd.isna(x): return ""
    return str(x).strip()

def load_image(row):
    path = DATA_ROOT / row["image_path"]
    return Image.open(path).convert("RGB")  # native resolution

def get_candidate_token_ids(tokenizer):
    cids = {}
    for letter in CHOICE_LETTERS:
        forms = [letter, " "+letter, "\n"+letter, letter+".", " "+letter+"."]
        ids = set()
        for f in forms:
            enc = tokenizer.encode(f, add_special_tokens=False)
            if enc: ids.add(enc[-1])
        cids[letter] = sorted(ids)
    return cids

def build_prompt(row, include_answer=False):
    choices = row["choices"]
    choices_text = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))
    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))
    lecture = clean_text(row.get("lecture", ""))
    prompt = "<image>\nYou are solving a science multiple-choice question.\n"
    prompt += "Use the image and text to choose the best answer.\n"
    prompt += "Answer with ONLY the letter and the full text of the correct choice.\n\n"
    if lecture: prompt += f"Lecture: {lecture}\n\n"
    if hint: prompt += f"Hint: {hint}\n\n"
    prompt += f"Question: {question}\n\nChoices:\n{choices_text}\n\nAnswer:"
    if include_answer:
        ans = int(row["answer"])
        prompt += f" {CHOICE_LETTERS[ans]}. {choices[ans]}"
    return prompt

def build_prompt_with_choices(row, shuffled_choices):
    choices_text = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(shuffled_choices))
    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))
    lecture = clean_text(row.get("lecture", ""))
    prompt = "<image>\nYou are solving a science multiple-choice question.\n"
    prompt += "Use the image and text to choose the best answer.\n"
    prompt += "Answer with ONLY the letter and the full text of the correct choice.\n\n"
    if lecture: prompt += f"Lecture: {lecture}\n\n"
    if hint: prompt += f"Hint: {hint}\n\n"
    prompt += f"Question: {question}\n\nChoices:\n{choices_text}\n\nAnswer:"
    return prompt

In [ ]:
# Load v2 checkpoint
processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "left"

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
    low_cpu_mem_usage=True, attn_implementation="eager",
)
model = PeftModel.from_pretrained(base_model, "/content/drive/MyDrive/lora_v2_best")
model.eval()
candidate_ids = get_candidate_token_ids(processor.tokenizer)
print("Loaded v2 checkpoint")

In [ ]:
# Inference functions
def predict_batch(df_batch):
    images = [load_image(row) for _, row in df_batch.iterrows()]
    prompts = [build_prompt(row) for _, row in df_batch.iterrows()]
    inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}
    with torch.inference_mode():
        logits = model(**inputs).logits[:, -1, :]
    log_probs = torch.log_softmax(logits, dim=-1)
    preds = []
    for i, (_, row) in enumerate(df_batch.iterrows()):
        scores = [log_probs[i, candidate_ids[CHOICE_LETTERS[ci]]].max().item()
                  for ci in range(len(row["choices"]))]
        preds.append(int(np.argmax(scores)))
    return preds

def predict_tta(df_batch, k=5, seed=42):
    rng = random.Random(seed)
    max_letters = len(CHOICE_LETTERS)
    batch_size = len(df_batch)
    accum = torch.zeros((batch_size, max_letters), dtype=torch.float32)
    rows = [row for _, row in df_batch.iterrows()]

    for pass_idx in range(k):
        perms = []
        for row in rows:
            n = len(row["choices"])
            if pass_idx == 0:
                perm = list(range(n))
            else:
                perm = list(range(n))
                rng.shuffle(perm)
            perms.append(perm)

        images = [load_image(row) for row in rows]
        prompts = []
        for row, perm in zip(rows, perms):
            shuffled_choices = [row["choices"][j] for j in perm]
            prompts.append(build_prompt_with_choices(row, shuffled_choices))

        inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
        inputs = {k_: v.to(model.device) if torch.is_tensor(v) else v for k_, v in inputs.items()}

        with torch.inference_mode():
            logits = model(**inputs).logits[:, -1, :]
        log_probs = torch.log_softmax(logits.float(), dim=-1).cpu()

        for i, (row, perm) in enumerate(zip(rows, perms)):
            n = len(row["choices"])
            for shuffled_pos, orig_choice in enumerate(perm):
                token_ids = candidate_ids[CHOICE_LETTERS[shuffled_pos]]
                score = max(log_probs[i, tid].item() for tid in token_ids)
                accum[i, orig_choice] += score

    for i, row in enumerate(rows):
        accum[i, len(row["choices"]):] = float("-inf")

    return accum.argmax(dim=1).tolist()

In [ ]:
# Validate: No TTA vs TTA k=5
model.eval()
gc.collect(); torch.cuda.empty_cache()

eval_df = val_df.copy().reset_index(drop=True)

print("Running no-TTA baseline...")
no_tta_preds = []
for start in tqdm(range(0, len(eval_df), BATCH_SIZE_INFER), desc="No TTA"):
    batch = eval_df.iloc[start:start+BATCH_SIZE_INFER]
    p = predict_batch(batch)
    no_tta_preds.extend(p)
    torch.cuda.empty_cache()

no_tta_acc = sum(p == a for p, a in zip(no_tta_preds, eval_df["answer"])) / len(eval_df)
print(f"No TTA: {no_tta_acc:.4f}")

print("\nRunning TTA (k=5)...")
tta_preds = []
for start in tqdm(range(0, len(eval_df), BATCH_SIZE_INFER), desc="TTA k=5"):
    batch = eval_df.iloc[start:start+BATCH_SIZE_INFER]
    p = predict_tta(batch, k=5)
    tta_preds.extend(p)
    torch.cuda.empty_cache()

tta_acc = sum(p == a for p, a in zip(tta_preds, eval_df["answer"])) / len(eval_df)
print(f"TTA k=5: {tta_acc:.4f}")
print(f"\nDifference: {tta_acc - no_tta_acc:+.4f}")
disagree = sum(a != b for a, b in zip(no_tta_preds, tta_preds))
print(f"Disagreements: {disagree}/{len(eval_df)}")

In [ ]:
# Submit TTA predictions
print("Generating TTA submission...")
all_preds = []
for start in tqdm(range(0, len(test_df), BATCH_SIZE_INFER), desc="TTA test"):
    batch = test_df.iloc[start:start+BATCH_SIZE_INFER]
    p = predict_tta(batch, k=5)
    all_preds.extend(p)
    torch.cuda.empty_cache()

submission = pd.DataFrame({"id": test_df["id"], "answer": all_preds})
submission.to_csv("/content/submission.csv", index=False)
submission.to_csv("/content/drive/MyDrive/submission_tta.csv", index=False)
print(f"Saved ({len(submission)} rows)")

from google.colab import files
files.download("/content/submission.csv")